This notebook evaluates the sentence classifier on the hold-out test set.

In [1]:
from dap_job_quality.pipeline.find_job_quality import JobQuality, split_into_chunks

from datasets import Dataset
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import time

job_quality = JobQuality()
job_quality.load()

2024-09-25 11:38:03,828 - datasets - INFO - PyTorch version 2.1.2 available.


/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-09-25 11:38:06,164 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2024-09-25 11:38:06,323 - root - INFO - Loading models and variables
2024-09-25 11:38:06,464 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rosie.oxbury/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rosie.oxbury/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


2024-09-25 11:38:06,633 - root - INFO - Downloading the model...
2024-09-25 11:38:54,348 - root - INFO - Loading the model and tokenizer...
2024-09-25 11:38:54,663 - aiobotocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


2024-09-25 11:38:55,011 - root - INFO - Calculating embeddings for 131 target phrases ...


Batches: 100%|██████████| 5/5 [00:00<00:00,  9.04it/s]


In [2]:
sets = ['train', 'val', 'test']

datasets = {}

for set in sets:
    print(f'Loading {set} set...')
    datasets[set] = pd.read_parquet(f's3://open-jobs-lake/job_quality/sentence_classifier/inputs/labelled/{set}_df_20240725.parquet')
    

# test_df = pd.read_parquet('s3://open-jobs-lake/job_quality/sentence_classifier/inputs/labelled/test_df_20240725.parquet')

Loading train set...
Loading val set...
Loading test set...


In [3]:
output_dict = {}

for set, df in datasets.items():
    
    df = df[df['sentence'].notna()] # there were 3 NAs in the training set! Not sure how this happened
    
    dataset = Dataset.from_pandas(df)

    start_time = time.time()
    predictions = job_quality.job_quality_classifier(
                dataset["sentence"], batch_size=job_quality.batch_size
            )

    elapsed_time = time.time() - start_time
    print(f"Time taken: {elapsed_time:.2f} seconds")

    labels = []
    pred_scores = []
    for pred in predictions:
        labels.append(pred["label"])
        pred_scores.append(pred["score"])

    df["job_quality_label"] = labels
    df["job_quality_prob"] = pred_scores
    
    # Rename 'LABEL_0' to 0 and 'LABEL_1' to 1 in the 'job_quality_label' column
    df['job_quality_label'] = df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})
    
    output_dict[set] = df

Time taken: 94.54 seconds


/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_59457/3769224282.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["job_quality_label"] = labels
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_59457/3769224282.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["job_quality_prob"] = pred_scores
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_59457/3769224282.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future versio

Time taken: 15.95 seconds


/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_59457/3769224282.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['job_quality_label'] = df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})


Time taken: 24.32 seconds


/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_59457/3769224282.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['job_quality_label'] = df['job_quality_label'].replace({'LABEL_0': 0, 'LABEL_1': 1})


In [4]:
classification_reports = {}

for set, df in output_dict.items():
    conf_matrix = confusion_matrix(df['label'], df['job_quality_label'])
    print(f'{set} set confusion matrix:')
    print(conf_matrix)
    classification_reports[set] = pd.DataFrame(classification_report(df['label'], df['job_quality_label'], output_dict=True))

train set confusion matrix:
[[514  56]
 [ 64 548]]
val set confusion matrix:
[[ 95   5]
 [ 15 100]]
test set confusion matrix:
[[106  17]
 [ 12 115]]


In [5]:
classification_reports['train']

,0,1,accuracy,macro avg,weighted avg
precision,0.889273,0.907285,0.898477,0.898279,0.898599
recall,0.901754,0.895425,0.898477,0.898590,0.898477
f1-score,0.895470,0.901316,0.898477,0.898393,0.898497
support,570.000000,612.000000,0.898477,1182.000000,1182.000000


In [6]:
classification_reports['val']

,0,1,accuracy,macro avg,weighted avg
precision,0.863636,0.952381,0.906977,0.908009,0.911104
recall,0.950000,0.869565,0.906977,0.909783,0.906977
f1-score,0.904762,0.909091,0.906977,0.906926,0.907077
support,100.000000,115.000000,0.906977,215.000000,215.000000


In [7]:
classification_reports['test']

,0,1,accuracy,macro avg,weighted avg
precision,0.898305,0.871212,0.884,0.884759,0.884542
recall,0.861789,0.905512,0.884,0.883650,0.884000
f1-score,0.879668,0.888031,0.884,0.883849,0.883916
support,123.000000,127.000000,0.884,250.000000,250.000000
